In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [2]:
load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 47/47 [00:18<00:00,  2.54it/s]


In [12]:
len(deals)

47

In [13]:
deals

[<HSN Electronics Deals: Up to 70% off + free shipping>,
 <Oukitel P5000 5,120Wh 2,200W Portable Power Station for $1289 + free shipping>,
 <Refurb Headphones at Best Buy: Up to 66% off + free shipping>,
 <TV Flash Deals at Walmart: Up to 64% off + free shipping>,
 <Samsung S90D 4K OLED Gaming TVs (2024): Up to $1,900 off + free shipping>,
 <Apple Product Deals at HSN: Get Deals on New, Refurb, and Pre-Owned + free shipping>,
 <Lax Gadgets 9-Outlet USB Swivel Surge Protector 2-Pack for $25 + free shipping>,
 <Electronics Deals at Walmart: Up to 65% off + free shipping w/ $35>,
 <Eco-Worthy 48V 280AH LiFePO4 Vertical Standing Backup Battery 2-Pack for $3,850 + free shipping>,
 <Letsfit E42 Bluetooth Smartwatch for $30 + free shipping>,
 <Apple iPhone 17 Series & Air at Best Buy: Up to $1,100 off w/ trade-in + free shipping>,
 <Samsung Galaxy Tab S10 Lite 10.9" Android Tablet: $80 off + up to an extra $150 off w/ trade-in + free shipping>,
 <Samsung ViewFinity S65TC 34" Ultrawide 1440p H

In [14]:
deals[44]

<Twin Trundle Bed w/ USB Station for $125 + free shipping>

In [15]:
deals[44].describe()

'Title: Twin Trundle Bed w/ USB Station for $125 + free shipping\nDetails: Apply promo code "USKUN26" to take an extra $26 off the price on the page and get a great deal on a twin trundle bed. Plus, it ships free from a US warehouse. Buy Now at AliExpress    Features  Power strip in the headboard includes 2 USB ports and 2 AC outlets Side storage\nFeatures: \nURL: https://www.dealnews.com/products/Likimio/Twin-Trundle-Bed-w-USB-Station/493799.html?iref=rss-c196'

In [16]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [17]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [18]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: HSN Electronics Deals: Up to 70% off + free shipping
Details: There are lots of great discounts in this electronics sale, and the deals are even better this weekend, as they'll all ship for free. Over 2,000 items are on offer, from big brands like Apple, Samsung, and HP, while laptops and computers discounted by up to $100, and power generators marked down by as muc

In [19]:
def get_recommendations():
    completion = openai.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    print(completion)
    result = completion.choices[0].message.parsed
    return result

In [20]:
result = get_recommendations()

ParsedChatCompletion[DealSelection](id='chatcmpl-CFjQjZoKcY3KSxbBCJXhVP8FFgWJ3', choices=[ParsedChoice[DealSelection](finish_reason='stop', index=0, logprobs=None, message=ParsedChatCompletionMessage[DealSelection](content='{\n  "deals": [\n    {\n      "product_description": "The Oukitel P5000 is a robust portable power station featuring an impressive 5120Wh capacity and a powerful 2200W AC Pure Sine Wave Inverter. Designed for high-efficiency energy usage, it supports lightning-fast 1800W AC input and includes a 1400W uninterruptible power supply for emergencies. It\'s also solar-compatible, with a solar input capacity of up to 1000W, making it an ideal choice for outdoor adventures or backup power needs.",\n      "price": 1289,\n      "url": "https://www.dealnews.com/Oukitel-P5000-5-120-Wh-2-200-W-Portable-Power-Station-for-1289-free-shipping/21767464.html?iref=rss-c142"\n    },\n    {\n      "product_description": "The Samsung S90C QN65S90CAFXZA is a stunning 65-inch 4K OLED Smart 

In [22]:
result

DealSelection(deals=[Deal(product_description="The Oukitel P5000 is a robust portable power station featuring an impressive 5120Wh capacity and a powerful 2200W AC Pure Sine Wave Inverter. Designed for high-efficiency energy usage, it supports lightning-fast 1800W AC input and includes a 1400W uninterruptible power supply for emergencies. It's also solar-compatible, with a solar input capacity of up to 1000W, making it an ideal choice for outdoor adventures or backup power needs.", price=1289.0, url='https://www.dealnews.com/Oukitel-P5000-5-120-Wh-2-200-W-Portable-Power-Station-for-1289-free-shipping/21767464.html?iref=rss-c142'), Deal(product_description="The Samsung S90C QN65S90CAFXZA is a stunning 65-inch 4K OLED Smart TV renowned for its exceptional picture quality and deep blacks. It integrates smart features that enable seamless access to streaming services and apps, enhancing your viewing experience. The TV boasts a sleek design that complements any living space, ensuring that i

In [21]:
len(result.deals)

5

In [23]:
result.deals[1]

Deal(product_description="The Samsung S90C QN65S90CAFXZA is a stunning 65-inch 4K OLED Smart TV renowned for its exceptional picture quality and deep blacks. It integrates smart features that enable seamless access to streaming services and apps, enhancing your viewing experience. The TV boasts a sleek design that complements any living space, ensuring that it's not only a technological marvel but also a stylish addition to your home.", price=1199.0, url='https://www.dealnews.com/TV-Flash-Deals-at-Walmart-Up-to-64-off-free-shipping/21767568.html?iref=rss-c142')

In [24]:
from agents.scanner_agent import ScannerAgent

In [27]:
agent = ScannerAgent()
result = agent.scan([])

100%|██████████| 47/47 [00:17<00:00,  2.61it/s]


In [33]:
for deal in result.deals:
    print(deal)
    print()

product_description='The Oukitel P5000 is a powerful portable power station boasting an impressive capacity of 5120Wh and a powerful 2200W AC pure sine wave inverter. This heavy-duty power source is ideal for various applications, whether for camping, outdoor events, or as a backup during emergencies. With lightning-fast 1800W AC input and the option for solar input up to 1000W, this gadget ensures your devices remain charged and ready to use. The inclusion of a 1400W uninterruptible power supply makes it a reliable choice for critical power needs.' price=1289.0 url='https://www.dealnews.com/Oukitel-P5000-5-120-Wh-2-200-W-Portable-Power-Station-for-1289-free-shipping/21767464.html?iref=rss-c142'

product_description="The Samsung S90C QN65S90CAFXZA is a 65-inch 4K OLED Smart TV that delivers stunning visuals with deep blacks and vibrant colors. Equipped with smart technology, this television provides access to your favorite streaming services, making it an entertainment hub for any room